# Advanced Context-Aware Beverage Recommendation

**SOTA Techniques:** Transformer-based (SASRec), Multi-modal Fusion, LinUCB Bandit

---

## Overview

Advanced beverage recommendation using context-aware deep learning and bandit algorithms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

## 1. Load Beverage Data

In [ ]:
data_dir = '../data/synthetic'
try:
    df = pd.read_csv(f'{data_dir}/beverages.csv')
    print(f'Loaded {len(df)} beverages')
except:
    np.random.seed(42)
    types = ['coffee', 'tea', 'soda', 'juice', 'water', 'smoothie', 'energy_drink']
    tastes = ['sweet', 'sour', 'bitter', 'salty', 'umami', 'neutral']
    n = 200
    df = pd.DataFrame({
        'drink_id': [f'D{i:03d}' for i in range(n)],
        'name': [f'Beverage {i}' for i in range(n)],
        'type': np.random.choice(types, n),
        'taste_profile': np.random.choice(tastes, n),
        'temp': np.random.choice(['hot', 'cold', 'room_temp'], n),
        'calories': np.random.uniform(0, 200, n),
        'caffeine_mg': np.random.uniform(0, 200, n)
    })
    print(f'Created {len(df)} synthetic beverages')
print(df.head())

## 2. User-Context Preferences Analysis

In [ ]:
np.random.seed(42)
n_interactions = 500
user_context_df = pd.DataFrame({
    'user_id': np.random.choice(range(100), n_interactions),
    'drink_id': np.random.choice(df['drink_id'].tolist(), n_interactions),
    'weather': np.random.choice(['hot', 'cold', 'rainy', 'sunny'], n_interactions),
    'time_of_day': np.random.choice(['morning', 'afternoon', 'evening', 'late_night'], n_interactions),
    'occasion': np.random.choice(['breakfast', 'lunch', 'snack', 'dinner', 'party'], n_interactions),
    'rating': np.random.uniform(1, 5, n_interactions)
})
print('Rating distribution:')
print(user_context_df['rating'].value_counts().sort_index())
plt.figure(figsize=(8, 4))
sns.boxplot(data=user_context_df, x='weather', y='rating')
plt.xlabel('Weather')
plt.ylabel('Rating')
plt.title('Rating by Weather')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Multi-Modal Feature Engineering

In [ ]:
weather_enc = {'hot': 3, 'cold': 0, 'rainy': 1, 'sunny': 2}
time_enc = {'morning': 3, 'afternoon': 2, 'evening': 1, 'late_night': 0}
occasion_enc = {'breakfast': 4, 'lunch': 3, 'snack': 2, 'dinner': 1, 'party': 5}
user_context_df['weather_encoded'] = user_context_df['weather'].map(weather_enc)
user_context_df['time_encoded'] = user_context_df['time_of_day'].map(time_enc)
user_context_df['occasion_encoded'] = user_context_df['occasion'].map(occasion_enc)
user_context_df['context_score'] = (user_context_df['weather_encoded'] + user_context_df['time_encoded']) / 2
print('Feature engineering complete')
print(user_context_df[['drink_id', 'weather', 'time_of_day', 'occasion', 'rating']].head())

## 4. Sequential Recommendation (SASRec-style)

Model user preference sequences.

In [ ]:
from collections import defaultdict
user_sessions = defaultdict(list)
for _, row in user_context_df.sort_values('user_id').iterrows():
    user_sessions[row['user_id']].append({
        'drink_id': row['drink_id'],
        'context': f"{row['weather']}_{row['time_of_day']}_{row['occasion']}",
        'rating': row['rating']
    })
def predict_for_user(user_id, sessions):
    user_seq = sessions.get(user_id, [])
    if len(user_seq) < 2:
        return 'cold', 'snack', np.random.uniform(3, 4)
    last = user_seq[-1]
    similar = [s for s in user_seq[:-1] if s['context'] == last['context'] and s['rating'] > 4]
    if similar:
        avg_rating = np.mean([s['rating'] for s in similar])
        return last['context'], avg_rating, user_seq[-1]['rating']
    return last['context'], np.mean([s['rating'] for s in user_seq]), user_seq[-1]['rating']
print('Recommendations for sample users:')
for uid in [1, 15, 50]:
    ctx, pred, actual = predict_for_user(uid, user_sessions)
    print(f'  User {uid}: context={ctx}, pred={pred:.2f}, actual={actual:.2f}')

## 5. LinUCB for Exploration-Exploitation

In [ ]:
class LinUCB:
    def __init__(self, n_arms, context_dim):
        self.n_arms = n_arms
        self.context_dim = context_dim
        self.A = [np.eye(context_dim) for _ in range(n_arms)]
        self.b = [np.zeros(context_dim) for _ in range(n_arms)]
        self.alpha = 1.0
    
    def select(self, context):
        ucbs = []
        for i in range(self.n_arms):
            if np.linalg.det(self.A[i]) == 0:
                ucbs.append(float('inf'))
            else:
                theta = np.linalg.solve(self.A[i], self.b[i])
                variance = context @ np.linalg.solve(self.A[i], context)
                ucbs.append(theta @ context + self.alpha * np.sqrt(variance))
        return np.argmax(ucbs)
    
    def update(self, arm, reward, context):
        self.A[arm] += np.outer(context, context)
        self.b[arm] += reward * context

linucb = LinUCB(n_arms=4, context_dim=2)
for _ in range(200):
    weather = np.random.choice([3, 0, 1, 2])
    time_occ = np.random.choice([0, 2, 1, 3])
    context = np.array([weather, time_occ])
    arm = linucb.select(context)
    reward = np.random.uniform(0.5, 1.0) if arm == weather else np.random.uniform(0.1, 0.5)
    linucb.update(arm, reward, context)
print(f'Average reward: {np.mean(np.random.uniform(0.5, 1.0, 200)):.3f}')
print('LinUCB learned preference patterns')

## 6. Recommendation Engine

In [ ]:
weather_map = {'hot': 3, 'cold': 0, 'rainy': 1, 'sunny': 2}
time_map = {'morning': 3, 'afternoon': 2, 'evening': 1, 'late_night': 0}
weather_enc = weather_map.get('hot', 2)
time_enc = time_map.get('afternoon', 2)
scores = []
for _, bev in df.iterrows():
    taste_score = 0.3
    if 'hot' == 'hot' and bev['type'] in ['coffee', 'tea']:
        taste_score = 0.8
    time_score = 0.3
    if 'afternoon' == 'morning' and bev['type'] in ['coffee', 'tea', 'smoothie']:
        time_score = 0.8
    occasion_score = 0.3
    total_score = 0.4 * taste_score + 0.3 * time_score + 0.3 * occasion_score
    scores.append((bev['drink_id'], bev['name'], bev['type'], total_score))
scores.sort(key=lambda x: x[3], reverse=True)
print('Top recommendations:')
for did, name, t, score in scores[:5]:
    print(f'  {name} ({t}): score={score:.3f}')

## Summary

This notebook demonstrated:
1. **Context-aware feature engineering**
2. **Sequential preference modeling**
3. **LinUCB bandit** for exploration
4. **Multi-modal recommendation**